# Storage and Compression - Interactive Companion
## CMSC 178IP - Digital Image Processing

This Jupyter notebook serves as an interactive companion to the **Storage and Compression Enhanced Presentation**. It includes hands-on activities, code demonstrations, and practical exercises to reinforce the concepts covered in the presentation.

### 📚 Topics Covered:
1. **Image Types and Representations**
2. **Storage Formats and File Sizes**
3. **Compression Algorithms**
4. **Interactive Activities and Experiments**

### 🎯 Learning Objectives:
- Understand different image types and their memory requirements
- Compare storage formats and their characteristics
- Implement basic compression algorithms
- Analyze compression trade-offs

---

## Setup and Dependencies

First, let's import all necessary libraries and set up our environment.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image, ImageDraw, ImageFont
import os
import io
from collections import Counter
import heapq
from scipy import ndimage
from skimage import data, filters, color
import ipywidgets as widgets
from IPython.display import display, Image as IPImage, HTML
import warnings
warnings.filterwarnings('ignore')

# Set up matplotlib for inline plots
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

print("✅ All libraries imported successfully!")
print("📚 Ready to explore Storage and Compression concepts!")

---
# Part 1: Image Types and Representations

Let's start by exploring different image types and understanding how they are represented in memory.

## 1.1 Creating Sample Images

In [ ]:
def create_sample_image(size=(256, 256)):
    """Create a sample image with various features for demonstration"""
    img = np.zeros((*size, 3), dtype=np.uint8)
    
    # Create gradient background
    for i in range(size[0]):
        for j in range(size[1]):
            img[i, j, 0] = int(255 * (i / size[0]))  # Red gradient
            img[i, j, 1] = int(255 * (j / size[1]))  # Green gradient
            img[i, j, 2] = int(255 * ((i + j) / (size[0] + size[1])))  # Blue gradient
    
    # Add geometric shapes
    cv2.circle(img, (64, 64), 30, (255, 255, 255), -1)
    cv2.rectangle(img, (150, 150), (200, 200), (0, 0, 0), -1)
    cv2.circle(img, (200, 64), 25, (255, 0, 0), 3)
    
    return img

# Create our sample image
sample_rgb = create_sample_image()

plt.figure(figsize=(10, 4))
plt.imshow(sample_rgb)
plt.title('Sample RGB Image for Demonstrations', fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"📸 Sample image created: {sample_rgb.shape}")
print(f"💾 Memory size: {sample_rgb.nbytes:,} bytes")

## 1.2 Image Type Conversions

Let's convert our sample image to different types and analyze their characteristics.

In [ ]:
def demonstrate_image_types(rgb_img):
    """Convert RGB image to different types and display them"""
    
    # 1. Binary Image
    gray = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2GRAY)
    _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    
    # 2. Grayscale Image (already computed above)
    
    # 3. RGB Image (original)
    
    # 4. Indexed Color (using PIL for better control)
    pil_img = Image.fromarray(rgb_img)
    indexed_img = pil_img.quantize(colors=16)  # Reduce to 16 colors
    indexed_array = np.array(indexed_img)
    
    # Calculate memory requirements
    binary_size = binary.nbytes
    gray_size = gray.nbytes
    rgb_size = rgb_img.nbytes
    indexed_size = indexed_array.nbytes + (16 * 3)  # palette size
    
    # Display results
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    # Top row: Images
    axes[0, 0].imshow(binary, cmap='gray')
    axes[0, 0].set_title('Binary Image\n(1-bit per pixel)', fontweight='bold')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(gray, cmap='gray')
    axes[0, 1].set_title('Grayscale Image\n(8-bit per pixel)', fontweight='bold')
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(rgb_img)
    axes[0, 2].set_title('RGB Color Image\n(24-bit per pixel)', fontweight='bold')
    axes[0, 2].axis('off')
    
    axes[0, 3].imshow(indexed_array, cmap='tab20')
    axes[0, 3].set_title('Indexed Color\n(4-bit + palette)', fontweight='bold')
    axes[0, 3].axis('off')
    
    # Bottom row: Memory analysis
    sizes = [binary_size, gray_size, rgb_size, indexed_size]
    labels = ['Binary', 'Grayscale', 'RGB', 'Indexed']
    colors = ['black', 'gray', 'blue', 'orange']
    
    for i in range(4):
        axes[1, i].bar([labels[i]], [sizes[i]], color=colors[i], alpha=0.7)
        axes[1, i].set_ylabel('Memory (bytes)')
        axes[1, i].set_title(f'{sizes[i]:,} bytes')
        axes[1, i].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return {
        'binary': binary,
        'grayscale': gray,
        'rgb': rgb_img,
        'indexed': indexed_array,
        'sizes': dict(zip(labels, sizes))
    }

# Demonstrate image types
image_types = demonstrate_image_types(sample_rgb)
print("\n📊 Memory Usage Comparison:")
for img_type, size in image_types['sizes'].items():
    print(f"  {img_type:10}: {size:6,} bytes ({size/image_types['sizes']['rgb']*100:.1f}% of RGB)")

## 🎯 Activity 1: Bit Depth Exploration

Let's create an interactive widget to explore how bit depth affects image quality and file size.

In [ ]:
def quantize_image(image, bit_depth):
    """Quantize image to specified bit depth"""
    levels = 2 ** bit_depth
    quantized = np.round(image / 256 * levels) * (256 / levels)
    return np.clip(quantized, 0, 255).astype(np.uint8)

def interactive_bit_depth(bit_depth=8):
    """Interactive function for bit depth exploration"""
    gray_img = cv2.cvtColor(sample_rgb, cv2.COLOR_RGB2GRAY)
    quantized = quantize_image(gray_img, bit_depth)
    
    levels = 2 ** bit_depth
    file_size = quantized.nbytes
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Display quantized image
    ax1.imshow(quantized, cmap='gray')
    ax1.set_title(f'{bit_depth}-bit Image\n({levels} gray levels)', fontweight='bold')
    ax1.axis('off')
    
    # Display histogram
    ax2.hist(quantized.flatten(), bins=min(levels, 50), alpha=0.7, color='blue')
    ax2.set_title(f'Histogram - {levels} unique values')
    ax2.set_xlabel('Pixel Intensity')
    ax2.set_ylabel('Frequency')
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"📊 Bit Depth: {bit_depth} bits")
    print(f"🎨 Gray Levels: {levels}")
    print(f"💾 File Size: {file_size:,} bytes")
    print(f"🔍 Quality: {'Excellent' if bit_depth >= 8 else 'Good' if bit_depth >= 4 else 'Poor'}")

# Create interactive widget
bit_depth_widget = widgets.interact(
    interactive_bit_depth,
    bit_depth=widgets.IntSlider(
        value=8,
        min=1,
        max=8,
        step=1,
        description='Bit Depth:',
        style={'description_width': 'initial'}
    )
)

print("🎮 Use the slider above to explore different bit depths!")

---
# Part 2: Storage Formats and File Sizes

Now let's explore different storage formats and analyze their characteristics.

## 2.1 Format Comparison Experiment

In [ ]:
def compare_formats(image, quality_levels=[10, 30, 60, 90]):
    """Compare different storage formats and their file sizes"""
    
    # Create temporary directory for files
    import tempfile
    temp_dir = tempfile.mkdtemp()
    
    formats_data = []
    
    try:
        # BMP (uncompressed)
        bmp_path = os.path.join(temp_dir, 'test.bmp')
        cv2.imwrite(bmp_path, cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
        bmp_size = os.path.getsize(bmp_path)
        formats_data.append(('BMP', 'Uncompressed', bmp_size, 100))
        
        # PNG (lossless)
        png_path = os.path.join(temp_dir, 'test.png')
        cv2.imwrite(png_path, cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
        png_size = os.path.getsize(png_path)
        formats_data.append(('PNG', 'Lossless', png_size, 100))
        
        # JPEG at different quality levels
        for quality in quality_levels:
            jpg_path = os.path.join(temp_dir, f'test_q{quality}.jpg')
            cv2.imwrite(jpg_path, cv2.cvtColor(image, cv2.COLOR_RGB2BGR), 
                       [cv2.IMWRITE_JPEG_QUALITY, quality])
            jpg_size = os.path.getsize(jpg_path)
            formats_data.append((f'JPEG Q{quality}', 'Lossy', jpg_size, quality))
        
    finally:
        # Clean up temporary files
        import shutil
        shutil.rmtree(temp_dir)
    
    # Create visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # File sizes chart
    formats = [item[0] for item in formats_data]
    sizes = [item[2]/1024 for item in formats_data]  # Convert to KB
    colors = ['red' if 'BMP' in f else 'blue' if 'PNG' in f else 'green' for f in formats]
    
    bars = ax1.bar(range(len(formats)), sizes, color=colors, alpha=0.7)
    ax1.set_xlabel('Format')
    ax1.set_ylabel('File Size (KB)')
    ax1.set_title('File Size Comparison by Format', fontweight='bold')
    ax1.set_xticks(range(len(formats)))
    ax1.set_xticklabels(formats, rotation=45, ha='right')
    ax1.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar, size in zip(bars, sizes):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{size:.1f} KB', ha='center', va='bottom', fontsize=9)
    
    # Compression ratio chart
    reference_size = formats_data[0][2]  # BMP size as reference
    compression_ratios = [item[2]/reference_size * 100 for item in formats_data]
    
    ax2.plot(range(len(formats)), compression_ratios, 'o-', linewidth=2, markersize=8)
    ax2.set_xlabel('Format')
    ax2.set_ylabel('Size relative to BMP (%)')
    ax2.set_title('Compression Efficiency', fontweight='bold')
    ax2.set_xticks(range(len(formats)))
    ax2.set_xticklabels(formats, rotation=45, ha='right')
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed results
    print("\n📊 Detailed Format Comparison:")
    print(f"{'Format':<12} {'Type':<12} {'Size (KB)':<10} {'vs BMP':<8} {'Quality':<8}")
    print("-" * 60)
    
    for i, (fmt, fmt_type, size, quality) in enumerate(formats_data):
        size_kb = size / 1024
        ratio = size / reference_size * 100
        print(f"{fmt:<12} {fmt_type:<12} {size_kb:<10.1f} {ratio:<8.1f}% {quality}%")
    
    return formats_data

# Run format comparison
format_results = compare_formats(sample_rgb)

## 🎯 Activity 2: JPEG Quality Interactive Exploration

In [ ]:
def interactive_jpeg_quality(quality=90):
    """Interactive JPEG quality comparison"""
    
    # Encode and decode with specified quality
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, encoded = cv2.imencode('.jpg', cv2.cvtColor(sample_rgb, cv2.COLOR_RGB2BGR), encode_param)
    decoded = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
    decoded_rgb = cv2.cvtColor(decoded, cv2.COLOR_BGR2RGB)
    
    # Calculate file size and quality metrics
    file_size = len(encoded) / 1024  # KB
    
    # Calculate PSNR (Peak Signal-to-Noise Ratio)
    mse = np.mean((sample_rgb.astype(float) - decoded_rgb.astype(float)) ** 2)
    if mse == 0:
        psnr = float('inf')
    else:
        psnr = 20 * np.log10(255.0 / np.sqrt(mse))
    
    # Display comparison
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Original image
    axes[0, 0].imshow(sample_rgb)
    axes[0, 0].set_title('Original Image', fontweight='bold')
    axes[0, 0].axis('off')
    
    # Compressed image
    axes[0, 1].imshow(decoded_rgb)
    axes[0, 1].set_title(f'JPEG Quality {quality}%\nFile Size: {file_size:.1f} KB', fontweight='bold')
    axes[0, 1].axis('off')
    
    # Difference image
    diff = np.abs(sample_rgb.astype(float) - decoded_rgb.astype(float))
    axes[0, 2].imshow(diff.astype(np.uint8))
    axes[0, 2].set_title(f'Difference (Artifacts)\nPSNR: {psnr:.1f} dB', fontweight='bold')
    axes[0, 2].axis('off')
    
    # Zoomed comparison (center region)
    center_slice = slice(80, 180), slice(80, 180)
    
    axes[1, 0].imshow(sample_rgb[center_slice])
    axes[1, 0].set_title('Original (Zoomed)', fontweight='bold')
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(decoded_rgb[center_slice])
    axes[1, 1].set_title('Compressed (Zoomed)', fontweight='bold')
    axes[1, 1].axis('off')
    
    axes[1, 2].imshow(diff[center_slice].astype(np.uint8))
    axes[1, 2].set_title('Difference (Zoomed)', fontweight='bold')
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Quality assessment
    if quality >= 90:
        quality_desc = "Excellent - Minimal artifacts"
    elif quality >= 70:
        quality_desc = "Good - Some artifacts visible"
    elif quality >= 50:
        quality_desc = "Fair - Noticeable artifacts"
    else:
        quality_desc = "Poor - Significant artifacts"
    
    print(f"📊 Quality Assessment: {quality_desc}")
    print(f"📁 File Size: {file_size:.1f} KB")
    print(f"📈 PSNR: {psnr:.1f} dB")
    print(f"🔍 Compression Ratio: {(sample_rgb.nbytes / len(encoded)):.1f}:1")

# Create interactive widget
jpeg_widget = widgets.interact(
    interactive_jpeg_quality,
    quality=widgets.IntSlider(
        value=90,
        min=10,
        max=100,
        step=10,
        description='JPEG Quality:',
        style={'description_width': 'initial'}
    )
)

print("🎮 Use the slider above to explore JPEG quality levels!")

---
# Part 3: Compression Algorithms

Let's implement and explore basic compression algorithms.

## 3.1 Huffman Encoding Implementation

In [ ]:
class HuffmanNode:
    def __init__(self, char=None, freq=0, left=None, right=None):
        self.char = char
        self.freq = freq
        self.left = left
        self.right = right
    
    def __lt__(self, other):
        return self.freq < other.freq

def build_huffman_tree(text):
    """Build Huffman tree from input text"""
    # Count frequencies
    freq_counter = Counter(text)
    
    # Create priority queue (min-heap)
    heap = [HuffmanNode(char, freq) for char, freq in freq_counter.items()]
    heapq.heapify(heap)
    
    # Build tree
    while len(heap) > 1:
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)
        
        merged = HuffmanNode(freq=left.freq + right.freq, left=left, right=right)
        heapq.heappush(heap, merged)
    
    return heap[0] if heap else None

def build_codes(root):
    """Build Huffman codes from tree"""
    if not root:
        return {}
    
    codes = {}
    
    def dfs(node, code=""):
        if node.char is not None:  # Leaf node
            codes[node.char] = code if code else "0"  # Handle single character case
        else:
            if node.left:
                dfs(node.left, code + "0")
            if node.right:
                dfs(node.right, code + "1")
    
    dfs(root)
    return codes

def huffman_encode(text):
    """Encode text using Huffman coding"""
    if not text:
        return "", {}, None
    
    # Build tree and codes
    root = build_huffman_tree(text)
    codes = build_codes(root)
    
    # Encode text
    encoded = "".join(codes[char] for char in text)
    
    return encoded, codes, root

def demonstrate_huffman(text="ABRACADABRA"):
    """Demonstrate Huffman encoding with visualization"""
    
    print(f"🔤 Original text: '{text}'")
    print(f"📏 Length: {len(text)} characters")
    
    # Count frequencies
    freq_counter = Counter(text)
    print(f"\n📊 Character Frequencies:")
    for char, freq in sorted(freq_counter.items()):
        print(f"  '{char}': {freq}")
    
    # Encode
    encoded, codes, root = huffman_encode(text)
    
    print(f"\n🔧 Huffman Codes:")
    for char, code in sorted(codes.items()):
        print(f"  '{char}': {code}")
    
    print(f"\n🔢 Encoded: {encoded}")
    
    # Calculate compression statistics
    original_bits = len(text) * 8  # ASCII encoding
    compressed_bits = len(encoded)
    compression_ratio = (1 - compressed_bits / original_bits) * 100
    
    print(f"\n📈 Compression Analysis:")
    print(f"  Original size: {original_bits} bits ({len(text)} chars × 8 bits)")
    print(f"  Compressed size: {compressed_bits} bits")
    print(f"  Compression ratio: {compression_ratio:.1f}%")
    print(f"  Space saved: {original_bits - compressed_bits} bits")
    
    # Visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Frequency chart
    chars = list(freq_counter.keys())
    freqs = list(freq_counter.values())
    
    ax1.bar(chars, freqs, color='steelblue', alpha=0.7)
    ax1.set_title('Character Frequencies', fontweight='bold')
    ax1.set_xlabel('Characters')
    ax1.set_ylabel('Frequency')
    ax1.grid(axis='y', alpha=0.3)
    
    # Compression comparison
    methods = ['Original\n(ASCII)', 'Huffman\nEncoded']
    sizes = [original_bits, compressed_bits]
    colors = ['red', 'green']
    
    bars = ax2.bar(methods, sizes, color=colors, alpha=0.7)
    ax2.set_title('Compression Comparison', fontweight='bold')
    ax2.set_ylabel('Size (bits)')
    
    for bar, size in zip(bars, sizes):
        ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{size} bits', ha='center', va='bottom')
    
    ax2.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return encoded, codes, compression_ratio

# Demonstrate Huffman encoding
encoded_result, huffman_codes, compression = demonstrate_huffman()

print(f"\n✅ Huffman encoding demonstration complete!")
print(f"🎯 Achieved {compression:.1f}% compression")

## 🎯 Activity 3: Custom Text Huffman Encoding

In [ ]:
# Interactive Huffman encoding for custom text
def interactive_huffman_demo():
    """Interactive widget for custom Huffman encoding"""
    
    text_input = widgets.Textarea(
        value='ABRACADABRA',
        placeholder='Enter text to encode...',
        description='Text:',
        layout=widgets.Layout(width='400px', height='100px')
    )
    
    encode_button = widgets.Button(
        description='Encode Text',
        button_style='success',
        layout=widgets.Layout(width='200px')
    )
    
    output = widgets.Output()
    
    def on_encode_click(b):
        with output:
            output.clear_output()
            text = text_input.value.strip().upper()
            if text:
                demonstrate_huffman(text)
            else:
                print("⚠️ Please enter some text to encode!")
    
    encode_button.on_click(on_encode_click)
    
    # Display widgets
    display(widgets.VBox([
        widgets.HTML("<h3>🎮 Interactive Huffman Encoding</h3>"),
        text_input,
        encode_button,
        output
    ]))

interactive_huffman_demo()

## 3.2 Run Length Encoding (RLE) Implementation

In [ ]:
def run_length_encode(data):
    """Simple Run Length Encoding implementation"""
    if not data:
        return []
    
    encoded = []
    current_char = data[0]
    count = 1
    
    for char in data[1:]:
        if char == current_char:
            count += 1
        else:
            encoded.append((count, current_char))
            current_char = char
            count = 1
    
    encoded.append((count, current_char))
    return encoded

def run_length_decode(encoded):
    """Decode RLE encoded data"""
    decoded = []
    for count, char in encoded:
        decoded.extend([char] * count)
    return decoded

def demonstrate_rle():
    """Demonstrate RLE with various examples"""
    
    examples = [
        "AAABBBBCCAAA",
        "WWWWWWWWBBBWWWWWWWWBBBWWWWWWWW",
        "ABCDEFG",  # Worst case
        "AAAAAAAAAAAA"  # Best case
    ]
    
    print("🔧 Run Length Encoding (RLE) Demonstration\n")
    
    results = []
    
    for i, text in enumerate(examples, 1):
        print(f"📝 Example {i}: '{text}'")
        
        # Encode
        encoded = run_length_encode(list(text))
        encoded_str = ''.join(f"{count}{char}" for count, char in encoded)
        
        # Decode to verify
        decoded = ''.join(run_length_decode(encoded))
        
        # Calculate compression
        original_size = len(text)
        compressed_size = len(encoded_str)
        compression_ratio = (1 - compressed_size / original_size) * 100
        
        print(f"   📤 Encoded: {encoded_str}")
        print(f"   📥 Decoded: {decoded}")
        print(f"   ✅ Match: {'Yes' if text == decoded else 'No'}")
        print(f"   📊 Original: {original_size} chars, Compressed: {compressed_size} chars")
        print(f"   📈 Compression: {compression_ratio:+.1f}% {'(Expansion!)' if compression_ratio < 0 else ''}")
        print()
        
        results.append((text, original_size, compressed_size, compression_ratio))
    
    # Visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Size comparison
    example_names = [f"Ex {i}" for i in range(1, len(examples) + 1)]
    original_sizes = [r[1] for r in results]
    compressed_sizes = [r[2] for r in results]
    
    x = np.arange(len(example_names))
    width = 0.35
    
    ax1.bar(x - width/2, original_sizes, width, label='Original', color='red', alpha=0.7)
    ax1.bar(x + width/2, compressed_sizes, width, label='RLE Compressed', color='blue', alpha=0.7)
    
    ax1.set_xlabel('Examples')
    ax1.set_ylabel('Size (characters)')
    ax1.set_title('RLE Size Comparison', fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(example_names)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    # Compression ratio
    compression_ratios = [r[3] for r in results]
    colors = ['green' if ratio > 0 else 'red' for ratio in compression_ratios]
    
    bars = ax2.bar(example_names, compression_ratios, color=colors, alpha=0.7)
    ax2.set_xlabel('Examples')
    ax2.set_ylabel('Compression Ratio (%)')
    ax2.set_title('RLE Compression Effectiveness', fontweight='bold')
    ax2.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    ax2.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar, ratio in zip(bars, compression_ratios):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., 
                height + (1 if height >= 0 else -3),
                f'{ratio:.1f}%', ha='center', 
                va='bottom' if height >= 0 else 'top')
    
    plt.tight_layout()
    plt.show()
    
    print("💡 Key Insights:")
    print("   • RLE works best with data containing repeated sequences")
    print("   • Can actually increase size for non-repetitive data")
    print("   • Commonly used for binary images and simple graphics")
    print("   • Often combined with other compression methods")
    
    return results

# Demonstrate RLE
rle_results = demonstrate_rle()

---
# Part 4: Practical Activities and Experiments

Let's engage in some hands-on activities to reinforce our understanding.

## 🎯 Activity 4: Image Processing Pipeline Simulator

In [ ]:
def image_processing_pipeline():
    """Interactive image processing pipeline simulator"""
    
    # Create widgets
    image_choice = widgets.Dropdown(
        options=[('Camera', 'camera'), ('Coins', 'coins'), ('Checkerboard', 'checkerboard'), ('Custom', 'custom')],
        value='camera',
        description='Source Image:'
    )
    
    image_type = widgets.Dropdown(
        options=[('RGB Color', 'rgb'), ('Grayscale', 'gray'), ('Binary', 'binary')],
        value='rgb',
        description='Image Type:'
    )
    
    storage_format = widgets.Dropdown(
        options=[('PNG (Lossless)', 'png'), ('JPEG High', 'jpeg_high'), ('JPEG Medium', 'jpeg_med'), ('JPEG Low', 'jpeg_low')],
        value='png',
        description='Storage Format:'
    )
    
    process_button = widgets.Button(
        description='Process Image',
        button_style='info',
        layout=widgets.Layout(width='200px')
    )
    
    output = widgets.Output()
    
    def process_image(b):
        with output:
            output.clear_output()
            
            # Get source image
            if image_choice.value == 'camera':
                img = data.camera()
                if len(img.shape) == 2:
                    img = np.stack([img] * 3, axis=-1)
            elif image_choice.value == 'coins':
                img = data.coins()
                img = np.stack([img] * 3, axis=-1)
            elif image_choice.value == 'checkerboard':
                img = data.checkerboard()
                img = np.stack([img] * 3, axis=-1) * 255
            else:  # custom
                img = sample_rgb
            
            # Resize for consistency
            img = cv2.resize(img, (256, 256))
            
            # Convert to specified type
            if image_type.value == 'gray':
                if len(img.shape) == 3:
                    processed_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
                    display_img = np.stack([processed_img] * 3, axis=-1)
                else:
                    processed_img = img
                    display_img = np.stack([img] * 3, axis=-1)
            elif image_type.value == 'binary':
                if len(img.shape) == 3:
                    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
                else:
                    gray = img
                _, processed_img = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
                display_img = np.stack([processed_img] * 3, axis=-1)
            else:  # rgb
                processed_img = img
                display_img = img
            
            # Simulate storage format compression
            temp_dir = tempfile.mkdtemp()
            
            try:
                if storage_format.value == 'png':
                    file_path = os.path.join(temp_dir, 'test.png')
                    cv2.imwrite(file_path, cv2.cvtColor(display_img.astype(np.uint8), cv2.COLOR_RGB2BGR))
                elif storage_format.value == 'jpeg_high':
                    file_path = os.path.join(temp_dir, 'test_high.jpg')
                    cv2.imwrite(file_path, cv2.cvtColor(display_img.astype(np.uint8), cv2.COLOR_RGB2BGR), 
                               [cv2.IMWRITE_JPEG_QUALITY, 95])
                elif storage_format.value == 'jpeg_med':
                    file_path = os.path.join(temp_dir, 'test_med.jpg')
                    cv2.imwrite(file_path, cv2.cvtColor(display_img.astype(np.uint8), cv2.COLOR_RGB2BGR), 
                               [cv2.IMWRITE_JPEG_QUALITY, 70])
                else:  # jpeg_low
                    file_path = os.path.join(temp_dir, 'test_low.jpg')
                    cv2.imwrite(file_path, cv2.cvtColor(display_img.astype(np.uint8), cv2.COLOR_RGB2BGR), 
                               [cv2.IMWRITE_JPEG_QUALITY, 30])
                
                file_size = os.path.getsize(file_path)
                
                # Load back for comparison
                loaded_img = cv2.imread(file_path)
                loaded_img = cv2.cvtColor(loaded_img, cv2.COLOR_BGR2RGB)
                
            finally:
                shutil.rmtree(temp_dir)
            
            # Calculate metrics
            memory_size = processed_img.nbytes
            compression_ratio = memory_size / file_size
            
            # Display results
            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            
            # Original
            axes[0].imshow(img if len(img.shape) == 3 else img, cmap='gray' if len(img.shape) == 2 else None)
            axes[0].set_title('Source Image', fontweight='bold')
            axes[0].axis('off')
            
            # Processed
            if image_type.value == 'gray' or image_type.value == 'binary':
                axes[1].imshow(processed_img, cmap='gray')
            else:
                axes[1].imshow(processed_img)
            axes[1].set_title(f'Processed ({image_type.value.upper()})', fontweight='bold')
            axes[1].axis('off')
            
            # Stored/Loaded
            axes[2].imshow(loaded_img)
            axes[2].set_title(f'After Storage ({storage_format.value.upper()})', fontweight='bold')
            axes[2].axis('off')
            
            plt.tight_layout()
            plt.show()
            
            # Print analysis
            print(f"\n📊 Processing Pipeline Analysis:")
            print(f"   🖼️  Source: {image_choice.value}")
            print(f"   🎨 Type: {image_type.value}")
            print(f"   💾 Format: {storage_format.value}")
            print(f"   📏 Dimensions: {processed_img.shape}")
            print(f"   🧠 Memory: {memory_size:,} bytes")
            print(f"   📁 File Size: {file_size:,} bytes")
            print(f"   📈 Compression: {compression_ratio:.1f}:1")
            
            if 'jpeg' in storage_format.value:
                quality_map = {'jpeg_high': 95, 'jpeg_med': 70, 'jpeg_low': 30}
                print(f"   🎯 JPEG Quality: {quality_map[storage_format.value]}%")
    
    process_button.on_click(process_image)
    
    # Display interface
    display(widgets.VBox([
        widgets.HTML("<h3>🔧 Image Processing Pipeline Simulator</h3>"),
        widgets.HBox([image_choice, image_type, storage_format]),
        process_button,
        output
    ]))

# Create pipeline simulator
image_processing_pipeline()

## 🎯 Activity 5: Compression Efficiency Challenge

In [ ]:
def compression_challenge():
    """Interactive compression challenge game"""
    
    # Test images with different characteristics
    test_images = {
        'Text Document': np.random.choice([0, 255], (100, 100), p=[0.7, 0.3]),
        'Gradient': np.linspace(0, 255, 10000).reshape(100, 100),
        'Random Noise': np.random.randint(0, 256, (100, 100)),
        'Checkerboard': np.kron(np.array([[0, 1], [1, 0]]), np.ones((50, 50))) * 255,
        'Smooth Photo': filters.gaussian(data.camera()[100:200, 100:200], sigma=2) * 255
    }
    
    print("🎮 Compression Efficiency Challenge!")
    print("Predict which compression method will work best for each image type.\n")
    
    results = {}
    
    for name, img in test_images.items():
        img = img.astype(np.uint8)
        
        print(f"🖼️  Testing: {name}")
        
        # Test different compression methods
        temp_dir = tempfile.mkdtemp()
        
        try:
            # Original size
            original_size = img.nbytes
            
            # PNG (lossless)
            png_path = os.path.join(temp_dir, 'test.png')
            cv2.imwrite(png_path, img)
            png_size = os.path.getsize(png_path)
            png_ratio = original_size / png_size
            
            # JPEG high quality
            jpg_path = os.path.join(temp_dir, 'test.jpg')
            cv2.imwrite(jpg_path, img, [cv2.IMWRITE_JPEG_QUALITY, 90])
            jpg_size = os.path.getsize(jpg_path)
            jpg_ratio = original_size / jpg_size
            
            # RLE simulation (for binary-like images)
            flat_img = img.flatten()
            rle_encoded = run_length_encode(flat_img.tolist())
            rle_size = len(rle_encoded) * 2  # Simplified: count + value pairs
            rle_ratio = original_size / rle_size if rle_size > 0 else 0
            
        finally:
            shutil.rmtree(temp_dir)
        
        # Store results
        results[name] = {
            'original': original_size,
            'png': (png_size, png_ratio),
            'jpeg': (jpg_size, jpg_ratio),
            'rle': (rle_size, rle_ratio)
        }
        
        # Find best method
        best_method = max([('PNG', png_ratio), ('JPEG', jpg_ratio), ('RLE', rle_ratio)], key=lambda x: x[1])
        
        print(f"   📊 Original: {original_size:,} bytes")
        print(f"   🟦 PNG: {png_size:,} bytes ({png_ratio:.1f}:1)")
        print(f"   🟨 JPEG: {jpg_size:,} bytes ({jpg_ratio:.1f}:1)")
        print(f"   🟩 RLE: {rle_size:,} bytes ({rle_ratio:.1f}:1)")
        print(f"   🏆 Best: {best_method[0]} ({best_method[1]:.1f}:1)\n")
    
    # Create comprehensive visualization
    fig, axes = plt.subplots(2, len(test_images), figsize=(20, 8))
    
    for i, (name, img) in enumerate(test_images.items()):
        # Display image
        axes[0, i].imshow(img, cmap='gray')
        axes[0, i].set_title(name, fontweight='bold')
        axes[0, i].axis('off')
        
        # Display compression ratios
        data = results[name]
        methods = ['PNG', 'JPEG', 'RLE']
        ratios = [data['png'][1], data['jpeg'][1], data['rle'][1]]
        colors = ['blue', 'orange', 'green']
        
        bars = axes[1, i].bar(methods, ratios, color=colors, alpha=0.7)
        axes[1, i].set_title(f'Compression Ratios')
        axes[1, i].set_ylabel('Ratio (X:1)')
        axes[1, i].grid(axis='y', alpha=0.3)
        
        # Highlight best method
        best_idx = ratios.index(max(ratios))
        bars[best_idx].set_edgecolor('red')
        bars[best_idx].set_linewidth(3)
        
        # Add value labels
        for bar, ratio in zip(bars, ratios):
            axes[1, i].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                           f'{ratio:.1f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # Educational insights
    print("🎓 Key Learning Points:")
    print("   📝 Text/Binary images: RLE works well for repeated patterns")
    print("   🌈 Smooth gradients: JPEG excels with smooth color transitions")
    print("   🎲 Random noise: Hard to compress with any method")
    print("   🏁 Checkerboard: Pattern-based, good for RLE")
    print("   📸 Photographs: JPEG optimized for natural images")
    
    return results

# Run compression challenge
challenge_results = compression_challenge()

---
# Part 5: Summary and Assessment

Let's wrap up with a summary of what we've learned and some assessment questions.

## 📚 Key Concepts Summary

In [ ]:
def create_summary_visualization():
    """Create a comprehensive summary visualization"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Image Types Memory Usage
    image_types = ['Binary\n(1-bit)', 'Grayscale\n(8-bit)', 'RGB Color\n(24-bit)', 'Indexed\n(4-bit + palette)']
    bits_per_pixel = [1, 8, 24, 4]
    colors = ['black', 'gray', 'blue', 'orange']
    
    bars1 = axes[0, 0].bar(image_types, bits_per_pixel, color=colors, alpha=0.7)
    axes[0, 0].set_title('Image Types - Bits per Pixel', fontweight='bold', fontsize=14)
    axes[0, 0].set_ylabel('Bits per Pixel')
    axes[0, 0].grid(axis='y', alpha=0.3)
    
    for bar, bits in zip(bars1, bits_per_pixel):
        axes[0, 0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                       f'{bits}', ha='center', va='bottom', fontweight='bold')
    
    # 2. Compression Methods Effectiveness
    methods = ['Huffman\nEncoding', 'Run Length\nEncoding', 'JPEG\n(DCT)', 'PNG\n(DEFLATE)']
    effectiveness = [85, 70, 90, 75]  # Typical effectiveness percentages
    method_colors = ['red', 'green', 'orange', 'blue']
    
    bars2 = axes[0, 1].bar(methods, effectiveness, color=method_colors, alpha=0.7)
    axes[0, 1].set_title('Compression Methods - Typical Effectiveness', fontweight='bold', fontsize=14)
    axes[0, 1].set_ylabel('Effectiveness (%)')
    axes[0, 1].set_ylim(0, 100)
    axes[0, 1].grid(axis='y', alpha=0.3)
    
    for bar, eff in zip(bars2, effectiveness):
        axes[0, 1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
                       f'{eff}%', ha='center', va='bottom', fontweight='bold')
    
    # 3. File Format Use Cases
    formats = ['BMP', 'PNG', 'JPEG', 'GIF']
    use_cases = ['System\nIcons', 'Web\nGraphics', 'Photography', 'Animation']
    format_colors = ['red', 'blue', 'green', 'purple']
    
    y_pos = range(len(formats))
    bars3 = axes[1, 0].barh(y_pos, [1]*len(formats), color=format_colors, alpha=0.7)
    axes[1, 0].set_title('File Formats - Primary Use Cases', fontweight='bold', fontsize=14)
    axes[1, 0].set_yticks(y_pos)
    axes[1, 0].set_yticklabels([f'{fmt}\n{use}' for fmt, use in zip(formats, use_cases)])
    axes[1, 0].set_xlabel('Relative Usage')
    axes[1, 0].set_xlim(0, 1.2)
    
    # Remove x-axis ticks for cleaner look
    axes[1, 0].set_xticks([])
    
    # 4. Quality vs File Size Trade-off
    quality_levels = [10, 30, 50, 70, 90, 100]
    file_sizes = [15, 25, 40, 60, 85, 100]  # Relative file sizes
    quality_scores = [30, 60, 75, 85, 95, 100]  # Quality scores
    
    axes[1, 1].plot(file_sizes, quality_scores, 'o-', linewidth=3, markersize=8, color='blue', label='Quality')
    axes[1, 1].set_xlabel('Relative File Size (%)')
    axes[1, 1].set_ylabel('Image Quality (%)')
    axes[1, 1].set_title('Quality vs File Size Trade-off', fontweight='bold', fontsize=14)
    axes[1, 1].grid(alpha=0.3)
    axes[1, 1].legend()
    
    # Add annotations for key points
    axes[1, 1].annotate('Sweet Spot', xy=(60, 85), xytext=(40, 95),
                       arrowprops=dict(arrowstyle='->', color='red', lw=2),
                       fontsize=12, fontweight='bold', color='red')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("📊 Storage and Compression - Key Takeaways")
    print("\n🎯 Image Types:")
    print("   • Binary: 1 bit/pixel - Text, line art")
    print("   • Grayscale: 8 bits/pixel - Medical, scientific")
    print("   • RGB: 24 bits/pixel - Photography, displays")
    print("   • Indexed: 4-8 bits + palette - Simple graphics")
    
    print("\n💾 Storage Formats:")
    print("   • BMP: Uncompressed, maximum quality")
    print("   • PNG: Lossless, good for graphics")
    print("   • JPEG: Lossy, excellent for photos")
    print("   • GIF: Limited colors, supports animation")
    
    print("\n🗜️ Compression Methods:")
    print("   • Huffman: Variable-length coding")
    print("   • RLE: Run-length encoding for patterns")
    print("   • DCT: Frequency domain (JPEG)")
    print("   • DEFLATE: LZ77 + Huffman (PNG)")
    
    print("\n⚖️ Trade-offs:")
    print("   • Quality vs File Size")
    print("   • Compression Speed vs Ratio")
    print("   • Lossless vs Lossy")
    print("   • Compatibility vs Efficiency")

create_summary_visualization()

## 🎯 Self-Assessment Quiz

In [ ]:
def create_assessment_quiz():
    """Interactive assessment quiz"""
    
    questions = [
        {
            'question': 'Which image type requires the most memory per pixel?',
            'options': ['Binary', 'Grayscale', 'RGB Color', 'Indexed'],
            'correct': 2,
            'explanation': 'RGB color images use 24 bits (3 bytes) per pixel, more than any other standard type.'
        },
        {
            'question': 'Which compression method is best for images with repeated patterns?',
            'options': ['Huffman Encoding', 'Run Length Encoding', 'DCT Transform', 'DEFLATE'],
            'correct': 1,
            'explanation': 'Run Length Encoding (RLE) excels at compressing data with repeated sequences.'
        },
        {
            'question': 'JPEG compression is considered:',
            'options': ['Lossless', 'Lossy', 'Variable', 'Perfect'],
            'correct': 1,
            'explanation': 'JPEG uses lossy compression, discarding some image information to achieve smaller file sizes.'
        },
        {
            'question': 'Which format is best for photographs?',
            'options': ['BMP', 'PNG', 'JPEG', 'GIF'],
            'correct': 2,
            'explanation': 'JPEG is optimized for photographic images with smooth color transitions.'
        },
        {
            'question': 'What does DCT stand for?',
            'options': ['Digital Color Transform', 'Discrete Cosine Transform', 'Data Compression Table', 'Dynamic Compression Type'],
            'correct': 1,
            'explanation': 'DCT (Discrete Cosine Transform) converts spatial data to frequency domain for JPEG compression.'
        }
    ]
    
    score = 0
    
    print("🎓 Self-Assessment Quiz")
    print("Test your understanding of storage and compression concepts!\n")
    
    for i, q in enumerate(questions, 1):
        print(f"Question {i}: {q['question']}")
        for j, option in enumerate(q['options']):
            print(f"  {chr(65+j)}. {option}")
        
        # Get user answer (in a real interactive environment, this would be a widget)
        print(f"\n✅ Correct Answer: {chr(65+q['correct'])}. {q['options'][q['correct']]}")
        print(f"💡 Explanation: {q['explanation']}\n")
        print("-" * 60)
    
    print("\n🎯 Quiz completed! In an interactive environment, this would track your score.")
    print("\n📚 Additional Study Resources:")
    print("   • Review the presentation slides for detailed explanations")
    print("   • Experiment with the interactive widgets above")
    print("   • Try different images with the processing pipeline")
    print("   • Practice implementing compression algorithms")

create_assessment_quiz()

---
# 🎯 Final Project Ideas

Here are some project ideas to further explore storage and compression concepts:

## Project Suggestions

### 1. **Custom Image Compressor**
- Implement a complete image compression pipeline
- Compare your results with standard formats
- Optimize for specific image types

### 2. **Format Converter Tool**
- Build a tool that converts between different image formats
- Include quality and size optimization options
- Add batch processing capabilities

### 3. **Compression Analyzer**
- Create a tool that analyzes images and recommends optimal formats
- Consider content type, quality requirements, and file size constraints
- Generate detailed compression reports

### 4. **Educational Visualization Tool**
- Build interactive demonstrations of compression algorithms
- Show step-by-step algorithm execution
- Include real-time parameter adjustment

### 5. **Image Quality Metrics**
- Implement various image quality assessment methods
- Compare compressed vs original images
- Develop perceptual quality metrics

---
# 📝 Conclusion

Congratulations! You have completed the **Storage and Compression Interactive Companion**. 

## What You've Learned:
- ✅ Different image types and their memory requirements
- ✅ Storage format characteristics and trade-offs
- ✅ Compression algorithm implementations and analysis
- ✅ Practical applications of storage and compression concepts
- ✅ Interactive exploration of quality vs file size trade-offs

## Next Steps:
1. **Practice**: Continue experimenting with the interactive widgets
2. **Explore**: Try the compression challenge with your own images
3. **Implement**: Build your own compression algorithms
4. **Apply**: Use these concepts in real-world projects

## Resources for Further Learning:
- Digital Image Processing textbooks
- Image compression research papers
- Open-source image processing libraries
- Computer vision and graphics courses

**Happy learning! 🚀**